#  Phil Job Applier Agent - Rachel Briskman & Sanila Chowdhury (Daedalus Devs)

## Team Report #1 (03/24) - Created a function that takes in user's resume and parse it to output the extracted info & uses Skyvern to see how the library works for filling in the fields of a job website

<img src="AI_AGENT_WORKFLOW.png" width="700" />

### Installs and Imports (uncomment and run if needed)
#### We left it commented so we can use "run all" in the notebook without wasting time

In [ ]:

# %pip install chromadb
# # %pip install "skyvern==1.0.24"
# %pip install skyvern
# # %playwright install --with-deps chromium
# # %playwright install chromium
# %pip install langchain langchain-huggingface
# %pip install python-dotenv

# print ("Done installing")

In [ ]:
# #installing dependencies for browser-use (an alternative to skyvern)
# %pip install browser-use playwright
# %pip install -U langchain-google-genai
# %pip install -U browser-use
# # %playwright install chromium

### Import API keys from .env file
#### (Note: This is for local use. Colab imports keys differently)

In [ ]:
import os
from dotenv import load_dotenv
import platform
import sys


# Sanila needs to set the working directory explicity (macOS) but it's not necessary on Windows or WSL.
# added this macro so we don't have to manually comment/uncommment os.chdir() every time.
IS_MAC = platform.system() == "Darwin"
print(f"Running on: {platform.system()}")


load_dotenv()


gemini_key = os.getenv("GEMINI_API_KEY")
skyvern_key = os.getenv("SKYVERN_API_KEY")
phil_key = os.getenv("PHIL-KEY")


os.environ["ENABLE_AUTH"] = "false"

# from skyvern.forge.sdk import SkyvernForge
from skyvern import Skyvern
import nest_asyncio
import asyncio

# Apply nest_asyncio to allow nested event loops in Colab
nest_asyncio.apply()

# 2. Map it to the environment variables Skyvern/LiteLLM looks for
os.environ["GEMINI_API_KEY"] = gemini_key
# os.environ["GOOGLE_API_KEY"] = gemini_key  # Some versions prefer this
# Optional: Force Skyvern to use a specific model
os.environ["LLM_KEY"] = "gemini/gemini-1.5-flash"
print(f"Key check: {'FOUND' if os.environ.get('GEMINI_API_KEY') else 'MISSING'}")
# print("Keys loaded and environment configured.")

Running on: Darwin
Key check: FOUND


### A website we're feeding into our agent to fill in the field (this website is used for testing purpose)

In [ ]:
url = "https://demo.applitools.com/"

### Agent takes in the filepath where the user's resume is stored and we prompt the agent to extract the info in a JSON that easy for both LLM and humans to read

In [11]:
import asyncio
import nest_asyncio
import os
from google import genai
from IPython.display import display, Markdown

nest_asyncio.apply()
client_genai = genai.Client(api_key=gemini_key)

def parse_resume(file_path, prompt="Please parse this resume and return its contents in a clear, readable format. Use sections with headers (like First Name, Last Name, Contact, Summary, Experience, Education, Skills) and plain text or bullet points — no JSON or code blocks."):
    """
    Uploads a PDF to Gemini and returns the model's analysis.
    Caches the result so Gemini is only called once per resume.
    """
    cache_path = file_path.replace(".pdf", "_cached.json")

    # delete old cache if it exists to force re-parse with new format
    # if os.path.exists(cache_path):
    #     os.remove(cache_path)
    #     print(f"Cleared old cache at {cache_path}.")

    if os.path.exists(cache_path):
        print(f"Using cached resume data from {cache_path}...")
        with open(cache_path, "r") as f:
            return f.read()

    # upload user's resume
    print(f"Uploading {file_path}...")
    uploaded_file = client_genai.files.upload(
        file=file_path,
        config={"display_name": "User_PDF"}
    )

    # feeds LLM the prompt and uploaded resume PDF
    response = client_genai.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=[uploaded_file, prompt]
    )

    # save to cache so we don't call Gemini again
    with open(cache_path, "w") as f:
        f.write(response.text)
    print(f"Resume parsed and cached to {cache_path}")

    return response.text

# call the function
# result = parse_resume("content/jakes-resume.pdf")
# display(Markdown(f"<div style='font-size: 14px'>{result}</div>"))

## Plans before next team report:
* #### Need to figure out how to see the filled in website, not as a screenshot but in another tab.
* #### Follow our workflow where the user can make corrections to the extracted resume info.

## Team Report #2 (03/31) - Created human in the loop for resume info verification, set up ChromaDB, and opened a window of the job website (dummy url) and filled in a few fields.
### (Replaced Skyvern with Browser_Use). This code will not run on Colab, only locally.

<img src="AI_AGENT_WORKFLOW_REPORT_2.png" width="700" />

### Developed a human in the loop where agent takes in the extracted info and ask the user if the info is correct. Until the user is satisifed, the agent will keep updating the extracted info based on user's feedback.

In [12]:
from IPython.display import display, Markdown

def verify_resume_info(extracted_info):
    current_info = extracted_info
    while True:
        print("\n" + "="*10)
        print("EXTRACTED RESUME INFO:")
        print("="*10)
        display(Markdown(f"<div style='font-size: 16px'>{current_info}</div>"))
        print("="*10)

        user_confirm = input("\nIs this information correct? (y/n): ").strip().lower()
        print(f"Is this information correct? (y/n): {user_confirm}")

        if user_confirm == "y":
            print("\nGreat! Moving forward with this information.")
            break
        else:
            correction = input("\nWhat needs to be corrected or added? Please describe: ").strip()
            print(f"What needs to be corrected or added? Please describe: {correction}")

            print("\nUpdating extracted info...")
            response = client_genai.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=[
                    f"""Here is the currently extracted resume information:
{current_info}
The user wants the following correction or addition:
{correction}
Please update the extracted resume JSON accordingly and return the full updated JSON."""
                ]
            )
            current_info = response.text
            print("\nInfo updated! Let's review it again.")
    return current_info


# call the function to store the extracted info
# result = parse_resume("content/jakes-resume.pdf")

# # run verify_resume_info to verify and correct
# final_info = verify_resume_info(result)

### Setting up ChromaDB which will store the correct parsed info of user's resume

In [13]:
import chromadb
import os
import hashlib
from dotenv import load_dotenv

# debug - check if values are loaded
print(f"CHROMA_HOST: {os.getenv('CHROMA_HOST')}")
print(f"CHROMA_TENANT: {os.getenv('CHROMA_TENANT')}")
print(f"CHROMA_DATABASE: {os.getenv('CHROMA_DATABASE')}")
print(f"CHROMA_API_KEY: {'FOUND' if os.getenv('CHROMA_API_KEY') else 'MISSING'}")

chroma_client = chromadb.HttpClient(
    ssl=True,
    host=os.getenv("CHROMA_HOST"),
    tenant=os.getenv("CHROMA_TENANT"),
    database=os.getenv("CHROMA_DATABASE"),
    headers={"x-chroma-token": os.getenv("CHROMA_API_KEY")}
)
collection = chroma_client.get_or_create_collection(name="resumes")
print(f"Connected to ChromaDB! Total resumes stored: {collection.count()}")

def store_resume_in_chromadb(resume_text, file_path, email):
    # use email as unique identifier so multiple resumes can be stored
    email = input("Please enter your email address (used as your unique ID): ").strip()
    doc_id = hashlib.md5(email.encode()).hexdigest()

    # Check if resume already exists
    existing = collection.get(ids=[doc_id])
    if existing["documents"]:
        print(f"⚠️ Resume already exists for {email}. Skipping upload.")
        return doc_id
    
    # Also check if same resume content already exists under a different email
    all_docs = collection.get(include=["documents", "metadatas"])
    for i, doc in enumerate(all_docs["documents"]):
        if doc == resume_text:
            existing_email = all_docs["metadatas"][i].get("email") or all_docs["ids"][i]
            print(f"⚠️ This resume already exists in ChromaDB under {existing_email}. Skipping upload.")
            return all_docs["ids"][i]

    collection.upsert(
        documents=[resume_text],
        ids=[doc_id],
        metadatas=[{"file_path": file_path, "email": email}]
    )
    print(f"✅ Resume stored! Email: {email} | ID: {doc_id}")
    return doc_id

# store the verified resume
# doc_id = store_resume_in_chromadb(final_info, "content/jakes-resume.pdf", "jake@su.edu")
print("Resume stored successfully!")

CHROMA_HOST: api.trychroma.com
CHROMA_TENANT: 7eab8b90-a03c-4bc3-9704-e5e4f8a3f82b
CHROMA_DATABASE: phil-job-agent
CHROMA_API_KEY: FOUND
Connected to ChromaDB! Total resumes stored: 6
Resume stored successfully!


## Plans for spring break:
* ##### Implementing a function where our agent to fill in the fields from the job website using Skyvern (we have started this process but currently experiencing errors)
* ##### Implementing a loop for our agent to handle when it comes to short-answer response
* ##### Have a prompt that will tell the LLM to generate answer based on the context given and the info it extracted from user's resume
* ##### Have a loop where as long as the answer is too vague, keep asking user for more info and implement a function for LLM to do web search (depending on the short answer question)
* ##### Have LLM generate a short answer response and fill out the field with Skyvern
* ##### IF WE FIND TIME -> Have an evalulation where user can give the LLM feedback at the end of the overall performance and store that in ChromaDB to help Phil improve

## Team Report #3 (04/14) - Resolved the EXHAUSTED API key issue by switching to OpenRouter (paid). Implemented most of the workflow and tested the agent.
### Started the evaluation portion of this project. The agent is able to open a window to the provided URL and fill in fields. 
### It successfully extracts the job title and the company name from the website. If the agent successfully fills out the entire application, it uses those fields to update the CSV file. (Works but unreliably because of the drop-down issue).


### See video-demo.mp4 at https://drive.google.com/file/d/1_tLs-5_bf-JU4mLIjEcZ-HX6wFeDz0mI/view?usp=sharing (It gets stuck on a drop down so I ended up interrupting the cell, which closed the browser.)
### (Video too big for git so I put it in Google Drive)


### Currently testing the agent with the job application at https://job-boards.greenhouse.io/attentive/jobs/4209296009

<img src="AI_AGENT_WORKFLOW_REPORT_3.png" width="700" />

### A function to write to a CSV file with details of the job application. If a CSV file does not exist, it will create one.

In [ ]:
import csv
import os

def save_application_history(url, company_name, time_applied, job_title):
    print ("Saving application history to application_history.csv...")
    csv_path = os.path.join(os.getcwd(), "application_history.csv")


    file_exists = os.path.isfile(csv_path)
    with open(csv_path, mode="a", newline="") as csvfile:
        fieldnames = ["url", "company_name", "time_applied", "job_title"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow({"url": url, "company_name": company_name, "time_applied": time_applied, "job_title": job_title})
    print(f"Application history updated: {company_name} - {job_title}")
    print(f"Saved to: {csv_path}")


# a function where we log the agent's report after each run, including the filled/empty fields and confidence rating for feedback and improvement purposes
def log_agent_report(report, name, job_title, url, filled_fields, empty_fields, agents_confidence_rating):
    print ("Saving agent report to agent_reports.csv...")
    csv_path = os.path.join(os.getcwd(), "agent_reports.csv")

    file_exists = os.path.isfile(csv_path)
    with open(csv_path, mode="a", newline="") as csvfile:
        fieldnames = ["name", "job_title", "report", "url", "filled_fields", "empty_fields", "agents_confidence_rating"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow({"name": name, "job_title": job_title, "report": report, "url": url, "filled_fields": filled_fields, "empty_fields": empty_fields, "agents_confidence_rating": agents_confidence_rating})
    print(f"Agent report logged for {name} - {job_title}")

### Agent ask user for any additional info that's not indicated on their resume but would be helpful in filling in the fields. That info are stored in ChromaDB.
### e.g. Demographic, Work authorization.

In [15]:
def collect_and_store_additional_info(email):
    additional_collection = chroma_client.get_or_create_collection(name="user_additional_info")
    doc_id = hashlib.md5(email.encode()).hexdigest()

    existing = additional_collection.get(ids=[doc_id])
    existing_data = {}
    if existing["documents"]:
        for line in existing["documents"][0].split("\n"):
            if ": " in line:
                key, val = line.split(": ", 1)
                existing_data[key.strip()] = val.strip()

    print("Collecting additional demographic and work authorization info...")
    print("Enter new values or press Enter to keep existing ones:")

    def prompt(label, key, default=""):
        current = existing_data.get(key, default)
        val = input(f"{label} (current: '{current}'): ").strip()
        return val if val else current

    gender = prompt("Gender (e.g. Male, Female, Non-binary, Prefer not to say)", "Gender")
    race = prompt("Race/Ethnicity (e.g. Asian, White, Hispanic, Black, Prefer not to say)", "Race/Ethnicity")
    veteran_status = prompt("Veteran status (e.g. Not a veteran, Veteran, Prefer not to say)", "Veteran Status")
    disability_status = prompt("Disability status (e.g. No disability, Has disability, Prefer not to say)", "Disability Status")
    work_authorization = prompt("Are you authorized to work in the US? (Yes/No)", "Work Authorization")
    sponsorship = prompt("Do you require sponsorship now or in the future? (Yes/No)", "Requires Sponsorship")
    visa_type = prompt("Visa type if applicable (e.g. H1B, F1 OPT, Green Card, Citizen, N/A)", "Visa Type")

    info_text = f"""Gender: {gender or 'Prefer not to say'}
Race/Ethnicity: {race or 'Prefer not to say'}
Veteran Status: {veteran_status or 'Prefer not to say'}
Disability Status: {disability_status or 'Prefer not to say'}
Work Authorization: {work_authorization or 'Yes'}
Requires Sponsorship: {sponsorship or 'No'}
Visa Type: {visa_type or 'N/A'}""".strip()

    additional_collection.upsert(
        documents=[info_text],
        ids=[doc_id],
        metadatas=[{"type": "additional_info", "email": email}]
    )
    print(f"✅ Additional info updated in ChromaDB!")
    return info_text

def get_additional_info(email):
    additional_collection = chroma_client.get_or_create_collection(name="user_additional_info")
    doc_id = hashlib.md5(email.encode()).hexdigest()
    results = additional_collection.get(ids=[doc_id])
    if results["documents"]:
        print(f"✅ Additional info found for {email}")
        return results["documents"][0]
    print(f"❌ No additional info found for {email}")
    return None

### Prints the additional user info stored in ChromaDB (used for testing purpose)

In [16]:
def print_additional_info():
    print("Displaying your additional demographic and work authorization info...")
    additional_collection = chroma_client.get_or_create_collection(name="user_additional_info")
    
    email = input("Enter your email: ").strip().lower()
    doc_id = hashlib.md5(email.encode()).hexdigest()
    
    results = additional_collection.get(ids=[doc_id])
    
    if not results["documents"]:
        print(f"❌ No additional info found for {email}.")
        return None
    
    info = results["documents"][0]
    print("\n" + "="*40)
    print("📋 YOUR STORED ADDITIONAL INFO:")
    print("="*40)
    print(info)
    print("="*40)
    return info
_ = print_additional_info()

Displaying your additional demographic and work authorization info...

📋 YOUR STORED ADDITIONAL INFO:
Gender: female
Race/Ethnicity: white
Veteran Status: not a veteran
Disability Status: no disability
Work Authorization: yes
Requires Sponsorship: no
Visa Type: N/A


### Agent handling short-answer questions field

In [17]:
# shows what the LLM is thinking when generating short answer responses
def stream_gemini(prompt: str) -> str:
    response = client_genai.models.generate_content_stream(
        model="gemini-2.5-flash-lite",
        contents=[prompt]
    )
    full_response = ""
    for chunk in response:
        if chunk.text:
            print(chunk.text, end="", flush=True)
            sys.stdout.flush()
            full_response += chunk.text
    print()
    return full_response.strip()


def short_answer_field(field_name: str, resume_text: str, job_url: str) -> str:
    # print(f"\n📝 Short answer field detected: '{field_name}'")
    print(f"🤔 Generating answer from resume...")
    answer = stream_gemini(
        f"Based on this resume, generate a concise answer for the job application field: '{field_name}'.\n"
        f"Resume:\n{resume_text}\n\n"
        f"If the resume does not have enough info to answer this field, respond with exactly: NEEDS_MORE_INFO\n"
        f"Otherwise respond with only the answer, nothing else."
    )

    if "NEEDS_MORE_INFO" in answer.upper():
        print(f"⚠️ Resume doesn't have enough info for '{field_name}'.")
        user_info = input(f"Please provide info for '{field_name}' (or press Enter to skip): ").strip()
        if not user_info:
            print(f"⏭️ Skipping '{field_name}'")
            return None

        print(f"🤔 Checking if answer is too vague...")
        vague = stream_gemini(
            f"Is this answer too vague to fill in a job application field called '{field_name}'? "
            f"Answer only YES or NO.\nAnswer: {user_info}"
        )
        while "YES" in vague.upper():
            print("⚠️ That answer is too vague.")
            user_info = input(f"Please provide a more specific answer for '{field_name}': ").strip()
            if not user_info:
                return None
            print(f"🤔 Re-checking vagueness...")
            vague = stream_gemini(
                f"Is this answer too vague to fill in a job application field called '{field_name}'? "
                f"Answer only YES or NO.\nAnswer: {user_info}"
            )

        print(f"🤔 Checking if company research is needed...")
        research_needed = stream_gemini(
            f"Does answering the job application field '{field_name}' require researching the company at {job_url}? "
            f"Answer only YES or NO."
        )
        extra_context = ""
        if "YES" in research_needed.upper():
            print(f"🔍 Researching company for '{field_name}'...")
            extra_context = stream_gemini(
                f"Summarize relevant info about the company at {job_url} "
                f"that would help answer the job application field: '{field_name}'."
            )
            print(f"✅ Company research done.")

        print(f"🤔 Generating final answer...")
        answer = stream_gemini(
            f"Generate a concise job application answer for the field '{field_name}'.\n"
            f"Resume:\n{resume_text}\n"
            f"User provided: {user_info}\n"
            f"Extra context: {extra_context}\n"
            f"Return only the answer, nothing else."
        )

    return answer

### Tracing the LLM, seeing the agent's thought as it go through the job application and filling out the field by setting browser-use's logging level to INFO (from WARNING) 

In [18]:
import logging

# This will show the Agent's "Thought", "Action", and "Observation" for every step
# Good for tracing.
import json
import re
from datetime import datetime
from browser_use import Agent, BrowserProfile
from browser_use.browser import BrowserSession
from browser_use.llm import ChatOpenAI

browser_logger = logging.getLogger('browser_use')

# Set browser-use logging level to INFO to see agent's Thought, Action, and Observation
logging.getLogger('browser_use').setLevel(logging.INFO)

# print out the current logging level. (should be INFO)
print(f"DEBUG: Updated browser-use logger level: {logging.getLevelName(logging.getLogger('browser_use').getEffectiveLevel())}")

DEBUG: Updated browser-use logger level: INFO


### Agent fills in the application field

In [19]:
import logging

# This will show the Agent's "Thought", "Action", and "Observation" for every step
# logging.basicConfig(level=logging.INFO)
# logging.getLogger('browser_use').setLevel(logging.INFO)
import json
import re
from datetime import datetime
from browser_use import Agent, BrowserProfile
from browser_use.browser import BrowserSession
from browser_use.llm import ChatOpenAI


async def fill_application(job_url: str, resume_text: str, additional_info: str = ""):
    print("Starting fill_application()\n")
    llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_KEY"),
        model="gemini-2.5-flash-lite",
    )

    browser_session = BrowserSession(
        browser_profile=BrowserProfile(headless=False, keep_alive=True)
    )
    await browser_session.start()

    # open browser to the job URL
    agent_open = Agent(
        task=f"Go to {job_url} and wait. Do not fill anything.",
        llm=llm,
        browser_session=browser_session,
        max_steps=3,
        use_vision=False,
        
    )
    await agent_open.run()
    print(f"Opened browser to {job_url}")
    # wait for user to sign in
    signed_in = input("Are you signed in (if needed)? (y/n): ").strip().lower()
    while signed_in != "y" and signed_in != "yes":
        signed_in = input("Please sign in and then enter y to continue: ").strip().lower()

    # ask user to upload resume manually
    input("Please upload your resume manually in the browser if required, then press Enter to continue...")
    print("Continuing...")

    # detect short answer fields, company name, and job title
    agent_detect = Agent(
        task=(
            f"IMPORTANT - USE THIS INFO FOR DEMOGRAPHIC AND AUTHORIZATION FIELDS. IF YOU ARE UNSURE OF WHAT TO DO, SIMPLE SKIP THE FIELD (mention it to the user) AND DO NOT GO BACK TO THAT SKIPPED FIELD (no matter what):\n"
            f"{additional_info}\n\n"
            f"The page {job_url} is already open. "
            "Look at the page and do the following:\n"
            "1. Find the company name and job title from the page.\n"
            "2. Find any SHORT ANSWER or ESSAY fields (e.g. 'Why do you want to work here?', 'Tell us about yourself', 'Cover letter', etc.).\n"
            "Return ONLY a JSON like this:\n"
            '{"company_name": "Google", "job_title": "Software Engineer", "short_answer_fields": ["field1", "field2"]}'
            "\nIf there are no short answer fields, return: "
            '{"company_name": "Google", "job_title": "Software Engineer", "short_answer_fields": []}'
        ),
        llm=llm,
        browser_session=browser_session,
        use_vision=True,
        max_steps=5,
        max_actions_per_step=3,
    )
    detect_result = await agent_detect.run()
    detect_final = detect_result.final_result()

    # handle short answer fields and extract company/job title
    short_answer_context = ""
    detected_company = "Unknown"
    detected_job_title = "Unknown"
    try:
        json_match = re.search(r'\{.*\}', detect_final, re.DOTALL)
        detected = json.loads(json_match.group()) if json_match else json.loads(detect_final)
        short_answer_fields = detected.get("short_answer_fields", [])
        detected_company = detected.get("company_name", "Unknown")
        detected_job_title = detected.get("job_title", "Unknown")
        print(f"🏢 Company: {detected_company} | 💼 Job Title: {detected_job_title}")

        print("\nStarting to fill the application form...")

        if short_answer_fields:
            print(f"\n📋 Short answer fields found: {short_answer_fields}")
            answers = {}
            for field in short_answer_fields:
                answer = short_answer_field(field, resume_text, job_url)
                if answer:
                    answers[field] = answer
            if answers:
                short_answer_context = "\n".join([f"{k}: {v}" for k, v in answers.items()])
    except Exception as e:
        print(f"⚠️ Could not detect page info: {e}")

    
    # fill all fields
    agent_fill = Agent(
        task=(
            f"IMPORTANT - USE THIS INFO FOR DEMOGRAPHIC AND AUTHORIZATION FIELDS:\n"
            f"{additional_info}\n\n"
            f"The page {job_url} is already open. "
            f"Use this resume to fill out the application:\n{resume_text}\n\n"
            + (f"Use these pre-generated answers for short answer fields:\n{short_answer_context}\n\n" if short_answer_context else "")
            + "Instructions:\n"
            "1. Find every empty field on the page (with the exception of file upload and dropdown fields.).\n"
            "2. If the field can be answered from the resume, fill it in.\n"
            "3. For DROPDOWN fields: SKIP ALL DROPDOWN FIELDS IMMEDIATELY. DO NOT ATTEMPT TO INTERACT WITH THEM.\n"
            "4. For SENSITIVE fields (Gender, Race, Veteran status, Disability status, Ethnicity): "
            "Use the additional user info provided above. If not found, pick the FIRST available option.\n"
            "5. For SPONSORSHIP/WORK AUTHORIZATION fields: Use info from additional info above. "
            "If unclear, select 'Yes' for authorization and 'No' for sponsorship.\n"
            "6. For COUNTRY fields: select 'United States' unless the resume says otherwise.\n"
            "7. For ALL FILE UPLOAD fields: DO NOT interact with them AT ALL. Skip immediately.\n"
            "8. For SHORT ANSWER or ESSAY fields: use the pre-generated answers provided above.\n"
            "9. DO NOT click 'Submit', 'Submit application', 'Apply', or ANY button that submits the form.\n"
            "10. You are allowed to go to the next page of the application (if there is one) after filling out the current page. BUT NEVER PRESS SUBMIT.\n"
            "11. Look back at the page and review your work. Rate your confidence 1-10. If you missed something, fix it.\n"
            "12. Final Action: Return ONLY a JSON object with this structure:\n"
            '{"applicant_name": "NAME", "filled": ["fields"], "empty": ["fields"], "skipped_uploads": ["files"], "company_name": "Name", "job_title": "Title", "confidence_rating": 1-10}'
        ),
        llm=llm,
        browser_session=browser_session,
        use_vision=True,
        max_steps=50,
        max_actions_per_step=10,
       include_attributes=[
            "id",           # form field targeting
            "name",         # input name → maps to field purpose (e.g. name="email")
            "type",         # input type (text, checkbox, radio, file, submit)
            "placeholder",  # hints what the field wants
            "aria-label",   # accessibility labels, common on ATS forms
            "value",        # current value / dropdown options
            "required",     # know which fields can't be skipped
            "href",         # for "Apply" / "Submit" links
            "role",         # identifies buttons, dialogs, dropdowns
            "data-testid",  # Greenhouse + Lever use these heavily for targeting
            "label",        # some ATS forms use <label> text to describe fields
            "class",        # last resort if nothing else identifies an element
        ]
    )

    result = await agent_fill.run()
    final = result.final_result()
    print("Agent result:", final)

   # parsing agent's JSON result to extract filled/empty fields and confidence rating for logging and feedback purposes
    parsed = {}
    try:
        json_match = re.search(r'\{.*\}', final, re.DOTALL)
        parsed = json.loads(json_match.group()) if json_match else json.loads(final)
    except Exception as e:
        if final and any(word in final.lower() for word in ["filled", "completed", "successfully"]):
            print("✅ Fields filled (no JSON returned by agent).")
        else:
            print(f"⚠️ Could not parse result: {e}")

    empty_fields = parsed.get("empty", [])
    filled_fields = parsed.get("filled", [])
    company_name = detected_company if detected_company != "Unknown" else parsed.get("company_name", "Unknown")
    job_title = detected_job_title if detected_job_title != "Unknown" else parsed.get("job_title", "Unknown")

    if empty_fields:
        print(f"⚠️ Empty fields remaining: {empty_fields}")
    else:
        print("✅ All fields filled!")
        # save_application_history(
            #     url=job_url,
            #     company_name=company_name,
            #     time_applied=datetime.today().strftime("%Y-%m-%d %H:%M:%S"),
            #     job_title=job_title,
        # )

    # save application on CSV and log agent report with parsed details
    save_application_history(
        url=job_url,
        company_name=company_name,
        time_applied=datetime.today().strftime("%Y-%m-%d %H:%M:%S"),
        job_title=job_title,
    )
    log_agent_report(
        report=final,
        name=os.getenv("USER_NAME", "Unknown User"),
        job_title=job_title,
        url=job_url,
        filled_fields=filled_fields,
        empty_fields=empty_fields,
        agents_confidence_rating=parsed.get("confidence_rating", "N/A")
    )

    return final

### Main function where user uses the agent to help apply to jobs

In [ ]:
import hashlib
async def apply_to_job_with_agent():
    resume_exists = input("Would you like to upload a new resume? Enter y if this is your first time (y/n): ")
    while resume_exists != "y" and resume_exists != "n":
        resume_exists = input("Invalid input. Please enter y or n: ")

    if resume_exists == "y":
        file_path = input("Please enter the path to your resume PDF: ").strip()
        extracted_info = parse_resume(file_path)
        verified_info = verify_resume_info(extracted_info)
        store_resume_in_chromadb(verified_info, file_path, email)
        resume_text = verified_info
    else:
        email = input("Please enter your email address: ").strip()
        
        doc_id = hashlib.md5(email.encode()).hexdigest()
        results = collection.get(ids=[doc_id], include=["documents"])
        if not results["documents"]:
            raise RuntimeError(f"No resume found for {email}. Please upload a resume first.")
        else:
            print("Using existing resume data. Moving on to job application...")
        resume_text = results["documents"][0]
        print(f"✅ Resume fetched for {email}")

    # Collect or fetch additional info using same email
    print("Fetching additional user info from ChromaDB...")
    additional_info = get_additional_info(email)
    if not additional_info:
        print("\n📋 No additional info found. Let's collect it now.")
        additional_info = collect_and_store_additional_info(email)
    else:
        update = input("\nWould you like to update your demographic/work authorization info? (y/n): ").strip().lower()
        if update == "y":
            additional_info = collect_and_store_additional_info(email)

    url = input("Please enter the URL of the job application you are applying to: ").strip()
    await fill_application(url, resume_text, additional_info)

await apply_to_job_with_agent()

## Plans before next team report:
* #### Continue working on evaluations:
    * ##### So far, we evaluated the component level of our agent where it parsed the resume correctly, leveraging LLM calls when needed (for additional info from the user and short-answer fields), and appending the filled-out job position to a CSV file.
    * ##### At the end-to-end level, we tested how our agent handles different types of fields (right now it's unreliable on drop-down field because it sometimes get stuck and keeps choosing random or incorrect options. It even keeps going back to the same field after filling it out so we would need to change our prompt), preventing the agent from pressing submit once all fields are filled, and how accurately it fills in fields with the correct information retrieved from the resume and the user's additional input.
    * ##### Next, we need to support multi-page job sites and navigation across different job boards (Greenhouse vs. Workday), as the agent currently only works on single-page applications.


* #### Short-answer field that involves HITL:
    * ##### We still need to integrate HITL, especially when the Agent lacks info for a field. We have already implemented basic short-answer handling, but need to extend it to asking the user for more info and triggering a web search when the question requires external or company-specific knowledge (e.g., "Why do you want to work at X?"), incorporating retrieved results into the generated response.

## Team Report #4 (04/21) - Evals
### In preparation for Evals, we fixed the ChromaDB code so it now properly stores multiple resumes. (Before, it just overwrote 1 resume each time)
### We implemented tracing the LLM to see the agent's response as it fill out the application field by setting browser-use's logging level to INFO from WARNING (the cell underneath where agent is handling short-answer fields)
### Created a function test_agent_with_dummy_resumes() to easily feed the agent multiple sites and resumes.
### Made the agent create a more detailed CSV to explain what it did. Similar to `save_application_history()`. Please see `log_agent_report()` (underneath `save_application_history()`)
### For more control, one of the sites we used is a google form we made. (https://forms.gle/sAtsVUrLm416zSwM9)

### Created a function test_agent_with_dummy_resumes() to easily feed the agent multiple sites and resumes. Dummy resumes were found on https://career.oregonstate.edu/sample-resumes

<img src="AI_AGENT_WORKFLOW_REPORT_4.png" width="700" />

In [20]:
#function takes in a list of single page urls and multi page urls.
#IMPORTANT: each resume in test_resumes/ folder should following this naming convention: dummy_resume_1.pdf  (only change the number)
async def test_agent_with_dummy_resumes(n_resumes, single_page_urls, multi_page_urls):
    # test_resumes_dir = "/Users/sanilachowdhury/Desktop/ai_agents/job-agent/content/test-resumes/"
    test_resumes_dir = os.path.join(os.getcwd(), "content/test-resumes/")
    for i in range(n_resumes):
        print(f"\n=== TESTING RESUME {i+1}/{n_resumes} ===")
        resume_path = f"{test_resumes_dir}dummy_resume_{i+1}.pdf"
        resume_text = parse_resume(resume_path)

        if (len(multi_page_urls) != 0):
            print("\nTesting on multi-page application...")
            await fill_application(multi_page_urls[i % len(multi_page_urls)], resume_text)

        if (len(single_page_urls) != 0):
            print("\nTesting on single-page application...")
            await fill_application(single_page_urls[i % len(single_page_urls)], resume_text)
        

    print(f"Done testing {n_resumes} resumes on single and multi-page applications.")


In [ ]:
single_page_test_urls = ["https://job-boards.greenhouse.io/attentive/jobs/4209296009"]
multi_page_test_urls = ["https://ssctech.wd1.myworkdayjobs.com/ssctechnologies/job/Waltham-MA---10-CityPoint/Software-Engineer-Intern---Co-op_R42466?utm_source=Simplify&ref=Simplify"]
await test_agent_with_dummy_resumes(2, single_page_test_urls, multi_page_test_urls)


=== TESTING RESUME 1/2 ===
Using cached resume data from /Users/sanilachowdhury/Desktop/ai_agents/job-agent/content/test-resumes/dummy_resume_1_cached.json...

Testing on multi-page application...
Starting fill_application()



2026-04-20T12:12:13.022167Z [info     ]  Using anonymized telemetry, see https://docs.browser-use.com/development/monitoring/telemetry. [browser_use.telemetry.service]
2026-04-20T12:12:13.038086Z [info     ]  🔗 Found URL in task: https://ssctech.wd1.myworkdayjobs.com/ssctechnologies/job/Waltham-MA---10-CityPoint/Software-Engineer-Intern---Co-op_R42466?utm_source=Simplify&ref=Simplify, adding as initial action... [browser_use.Agent🅰 38cb ⇢ 🅑 27bf 🅣 FC]
2026-04-20T12:12:13.212496Z [info     ]  🎯 Task: Go to https://ssctech.wd1.myworkdayjobs.com/ssctechnologies/job/Waltham-MA---10-CityPoint/Software-Engineer-Intern---Co-op_R42466?utm_source=Simplify&ref=Simplify and wait. Do not fill anything. [browser_use.Agent🅰 38cb ⇢ 🅑 27bf 🅣 FC]
2026-04-20T12:12:13.364122Z [info     ]  Starting a browser-use agent with version 0.12.6, with provider=openai and model=gemini-2.5-flash-lite [browser_use.Agent🅰 38cb ⇢ 🅑 27bf 🅣 FC]
2026-04-20T12:12:13.366427Z [info     ]    ▶️   navigate: url: https://sscte

Opened browser to https://ssctech.wd1.myworkdayjobs.com/ssctechnologies/job/Waltham-MA---10-CityPoint/Software-Engineer-Intern---Co-op_R42466?utm_source=Simplify&ref=Simplify


2026-04-20T12:13:28.049422Z [info     ]  🔗 Found URL in task: https://ssctech.wd1.myworkdayjobs.com/ssctechnologies/job/Waltham-MA---10-CityPoint/Software-Engineer-Intern---Co-op_R42466?utm_source=Simplify&ref=Simplify, adding as initial action... [browser_use.Agent🅰 5992 ⇢ 🅑 27bf 🅣 FC]
2026-04-20T12:13:28.051211Z [info     ]  🎯 Task: IMPORTANT - USE THIS INFO FOR DEMOGRAPHIC AND AUTHORIZATION FIELDS. IF YOU ARE UNSURE OF WHAT TO DO, SIMPLE SKIP THE FIELD (mention it to the user) AND DO NOT GO BACK TO THAT SKIPPED FIELD (no matter what):


The page https://ssctech.wd1.myworkdayjobs.com/ssctechnologies/job/Waltham-MA---10-CityPoint/Software-Engineer-Intern---Co-op_R42466?utm_source=Simplify&ref=Simplify is already open. Look at the page and do the following:
1. Find the company name and job title from the page.
2. Find any SHORT ANSWER or ESSAY fields (e.g. 'Why do you want to work here?', 'Tell us about yourself', 'Cover letter', etc.).
Return ONLY a JSON like this:
{"company_name": "G

Continuing...


...

🏢 Company: Attentive | 💼 Job Title: AI Intern

Starting to fill the application form...


2026-04-20T12:26:34.201854Z [info     ]  Starting a browser-use agent with version 0.12.6, with provider=openai and model=gemini-2.5-flash-lite [browser_use.Agent🅰 6479 ⇢ 🅑 de3f 🅣 61]
2026-04-20T12:26:34.328049Z [info     ]  
                              [browser_use.Agent🅰 6479 ⇢ 🅑 de3f 🅣 61]
2026-04-20T12:26:34.328363Z [info     ]  📍 Step 1:                      [browser_use.Agent🅰 6479 ⇢ 🅑 de3f 🅣 61]
2026-04-20T12:26:38.303874Z [info     ]    👍 Eval: The agent has successfully loaded the job application page and is ready to proceed with filling out the form. [browser_use.Agent🅰 6479 ⇢ 🅑 de3f 🅣 61]
2026-04-20T12:26:38.304880Z [info     ]    🧠 Memory: The user has provided a resume and specific instructions for filling out a job application form on the provided URL. The current page is the job description page, and the 'Apply' button needs to be clicked to access the form. [browser_use.Agent🅰 6479 ⇢ 🅑 de3f 🅣 61]
2026-04-20T12:26:38.305406Z [info     ]    🎯 Next goal: Click the 'Apply

Agent result: The job application form has been filled out according to the provided resume and instructions. All fields have been completed, with the exception of file uploads which were intentionally skipped. Demographic information has been provided as instructed, and the consent checkbox has been checked. The 'Website' field was left empty as no information was found in the resume.

Applicant Name: Alex Nguyen
Filled Fields: First Name, Last Name, Preferred First Name, Email, Phone, LinkedIn Profile, Country, Sponsorship, Gender, Race, Veteran status, Consent
Empty Fields: Website
Skipped Uploads: Resume/CV, Cover Letter
Company Name: Attentive
Job Title: Unknown (not provided in the prompt, assuming from context of job board)
Confidence Rating: 10
✅ Fields filled (no JSON returned by agent).
✅ All fields filled!
Saving application history to application_history.csv...
Application history updated: Attentive - AI Intern
Saved to: /Users/sanilachowdhury/Desktop/ai_agents/job-agent/ap

2026-04-20T12:28:04.004247Z [info     ]  🔗 Found URL in task: https://ssctech.wd1.myworkdayjobs.com/ssctechnologies/job/Waltham-MA---10-CityPoint/Software-Engineer-Intern---Co-op_R42466?utm_source=Simplify&ref=Simplify, adding as initial action... [browser_use.Agent🅰 95e4 ⇢ 🅑 1bfc 🅣 90]
2026-04-20T12:28:04.005524Z [info     ]  🎯 Task: Go to https://ssctech.wd1.myworkdayjobs.com/ssctechnologies/job/Waltham-MA---10-CityPoint/Software-Engineer-Intern---Co-op_R42466?utm_source=Simplify&ref=Simplify and wait. Do not fill anything. [browser_use.Agent🅰 95e4 ⇢ 🅑 1bfc 🅣 90]
2026-04-20T12:28:04.195551Z [info     ]  Starting a browser-use agent with version 0.12.6, with provider=openai and model=gemini-2.5-flash-lite [browser_use.Agent🅰 95e4 ⇢ 🅑 1bfc 🅣 90]
2026-04-20T12:28:04.197597Z [info     ]    ▶️   navigate: url: https://ssctech.wd1.myworkdayjobs.com/ssctechnologies/job/Waltham-MA---10-CityPoint/Software-Engineer-Intern---Co-op_R42466?utm_source=Simplify&ref=Si..., new_tab: False [browser_us

Opened browser to https://ssctech.wd1.myworkdayjobs.com/ssctechnologies/job/Waltham-MA---10-CityPoint/Software-Engineer-Intern---Co-op_R42466?utm_source=Simplify&ref=Simplify


## What we learned:
* #### Since the LLM does most of the "thinking" and work, most of the issues lie in `fill_application()`.
* #### Checkboxes, text fields, and calendar fields work great. 
* #### However, the agent frequently gets stuck on drop-down fields (picking some option repeatedly), despite the current prompt explicitly telling it skip drop-down fields. We added that prompt so it could at least continue to other fields in the mean time.
* #### The agent struggles with multi-page applications, even though the prompt allows the agent to proceed to the next page so long as the button does not say "Submit" or "Apply" or anything along the lines of "FINAL button for this application." Sometimes the agent goes to the next page, and sometimes it doesn't. We tested the agent with the same site a couple times. 

## Plans before next team report:
* ### Finishing up end-to-end level evals:
  * #### Continue evaluating and improving multi-page application support. As mentioned, the agent inconsistently navigates to the next page, and we want to make this reliable across different job sites (Greenhouse vs. Workday)
  * #### Fix dropdown field handling. It could be the way we prompted the agent in the `fill_application()` function when handling dropdown fields. Right now, we prompted the agent to skip those fields and not interact with them
  * #### Fix the format of agent_reports.csv. Right now, it's showing unknown user, [] for filled/empty fields, and N/A for confidence ratings. Not sure why that's the case since it worked before
* ### Move onto component level evals:
  * #### Just how we did tracing for end-to-end levels, we will apply the same for component levels (agent parsing resume, leveraging LLM calls, format to CSV files, and having user submit the application)